In [2]:
import os

print(os.getcwd())
print(os.path.exists('../data/processed/features.csv'))
print(os.path.exists('data/processed/features.csv'))

C:\Users\SATYA\OneDrive\Desktop\ContextFusion-PM\src
True
False


In [3]:
pd.read_csv('../data/processed/features.csv')

,air_temp_k,proc_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,target,TWF,HDF,PWF,OSF,...,tool_wear_min_roll_std_5,tool_wear_min_roll_max_5,tool_wear_min_roll_mean_10,tool_wear_min_roll_std_10,tool_wear_min_roll_max_10,tool_wear_min_roll_mean_20,tool_wear_min_roll_std_20,tool_wear_min_roll_max_20,temp_diff,power_proxy
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,...,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,10.5,66382.8
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,...,2.121320,3.0,1.500000,2.121320,3.0,1.500000,2.121320,3.0,10.5,65190.4
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,...,2.516611,5.0,2.666667,2.516611,5.0,2.666667,2.516611,5.0,10.4,74001.2
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,...,2.986079,7.0,3.750000,2.986079,7.0,3.750000,2.986079,7.0,10.4,56603.5
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,...,3.492850,9.0,4.800000,3.492850,9.0,4.800000,3.492850,9.0,10.5,56320.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,298.8,308.4,1604,29.5,14,0,0,0,0,0,...,3.492850,14.0,68.200000,97.947605,213.0,128.450000,91.634557,213.0,9.6,47318.0
9996,298.9,308.4,1632,31.8,17,0,0,0,0,0,...,3.492850,17.0,49.200000,85.692992,213.0,120.450000,94.123365,213.0,9.5,51897.6
9997,299.0,308.6,1645,33.4,22,0,0,0,0,0,...,4.690416,22.0,30.400000,64.496684,213.0,112.600000,95.519852,213.0,9.6,54943.0
9998,299.0,308.7,1408,48.5,25,0,0,0,0,0,...,5.431390,25.0,11.600000,8.099383,25.0,104.800000,96.008552,213.0,9.7,68288.0


In [4]:
def fuse_context():
    iot = pd.read_csv('../data/processed/features.csv')

    ctx = pd.read_csv(
        '../data/external/context_signals.csv',
        parse_dates=['timestamp']
    )

    ctx_cols = [
        'ambient_temp_c',
        'factory_load_pct',
        'humidity_pct'
    ]

    iot = iot.copy()

    for col in ctx_cols:
        iot[col] = ctx[col].values

    for col in ctx_cols:
        iot[f'{col}_roll_mean_5'] = (
            iot[col].rolling(5, min_periods=1).mean()
        )

    iot.to_csv('../data/processed/fused.csv', index=False)

    print(f"Fused dataset: {iot.shape}")

    return iot


fused_df = fuse_context()

Fused dataset: (10000, 65)


In [5]:
import os
print(os.path.exists('../data/processed/fused.csv'))

True


In [6]:
fused_df.head()

,air_temp_k,proc_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,target,TWF,HDF,PWF,OSF,...,tool_wear_min_roll_std_20,tool_wear_min_roll_max_20,temp_diff,power_proxy,ambient_temp_c,factory_load_pct,humidity_pct,ambient_temp_c_roll_mean_5,factory_load_pct_roll_mean_5,humidity_pct_roll_mean_5
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,...,0.000000,0.0,10.5,66382.8,15.802895,69.568908,64.964990,15.802895,69.568908,64.964990
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,...,2.121320,3.0,10.5,65190.4,13.782312,57.721724,71.347098,14.792603,63.645316,68.156044
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,...,2.516611,5.0,10.4,74001.2,20.126770,73.534856,67.711022,16.570659,66.941829,68.007703
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,...,2.986079,7.0,10.4,56603.5,17.032849,74.623132,61.443841,16.686206,68.862155,66.366738
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,...,3.492850,9.0,10.5,56320.0,14.384230,69.733240,64.019429,16.225811,69.036372,65.897276


In [7]:
fused_df['target'].value_counts()

target
0    9661
1     339
Name: count, dtype: int64

In [8]:
fused_df['target'].value_counts(normalize=True) * 100

target
0    96.61
1     3.39
Name: proportion, dtype: float64

In [9]:
X = fused_df.drop('target', axis=1)
y = fused_df['target']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [12]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.998
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1932
           1       1.00      0.94      0.97        68

    accuracy                           1.00      2000
   macro avg       1.00      0.97      0.98      2000
weighted avg       1.00      1.00      1.00      2000

[[1932    0]
 [   4   64]]
